In [ ]:
from ball_environment import *

env = GridWorld(10)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# net = SequentialVAE(latent_dim=5, permanent=True).to(device)
# net = SequentialVQVAE(3, 4, 1, permanent=True).to(device)
net = EMGFlowNet(state_size=3, dict_size=4, permanent=True)
if net._get_name() != "EMGFlowNet":
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
batch = 16
repetitions = 100
dec_losses, kl_losses = [], []

for epoch in (pbar := tqdm(range(4_000))):
    observations, actions = np.zeros((batch, repetitions+1, 11)), np.zeros((batch, repetitions))
    for gg in range(batch):
        env.reset()
        agent = Agent(type=[1, 2, 5][np.random.randint(3)])
        observations[gg, 0] = env.get_array_obs()
        for t in range(repetitions):
            action = agent.act(env)
            env.step(action)
            observations[gg, t+1] = env.get_array_obs()
            actions[gg, t] = action
    observations = torch.from_numpy(observations).float().to(device)
    actions = torch.from_numpy(actions).float().unsqueeze(-1)
    actions = torch.cat([torch.zeros(batch, 1, 1), actions], dim=1).to(device)
    if net._get_name() == "SequentialVAE":
        dec_loss, kl_loss, _, _ = net(observations, actions)
        loss = dec_loss + 0.05 * kl_loss
        dec_losses.append(dec_loss.item())
        kl_losses.append(kl_loss.item())
    if net._get_name() == "SequentialVQVAE":
        dec_loss, q_loss, _, _, _, _ = net(observations, actions)
        loss = dec_loss + q_loss
        dec_losses.append(dec_loss.item())
        kl_losses.append(q_loss.item())
    if net._get_name() == "EMGFlowNet":
        if epoch == 1_000:
            net.gfn_optimizer.param_groups[0]["lr"] = 0.00005
            net.decoder_optimizer.param_groups[0]["lr"] = 0.0003
        if (epoch % 200) < 100:
            gfn_loss = net.train_gfn(observations, actions, rand_prob=0.3)
            kl_losses.append(gfn_loss)
            pbar.set_description(f"GFN Loss: {kl_losses[-1]:.4f}")
        else:
            dec_loss = net.train_decoder(observations, actions, prob_exponent=-1)
            dec_losses.append(dec_loss)
            pbar.set_description(f"Dec Loss: {dec_losses[-1]:.4f}")
    if net._get_name() != "EMGFlowNet":
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        pbar.set_description(f"Dec Loss: {dec_losses[-1]:.4f}, KL Loss: {kl_losses[-1]:.4f}")

In [ ]:
batch = 500
observations, actions = np.zeros((batch, repetitions+1, 11)), np.zeros((batch, repetitions))
labels = torch.zeros((batch, repetitions)).long()
for gg in range(batch):
    env.reset()
    x = np.random.randint(3)
    agt = [1, 2, 5][x] # np.random.randint(1, 4)
    agent = Agent(type=agt)
    observations[gg, 0] = env.get_array_obs()
    # labels[gg] = x #agt - 1
    for t in range(repetitions):
        labels[gg, t] = agent.type - 1
        action = agent.act(env)
        env.step(action)
        observations[gg, t+1] = env.get_array_obs()
        actions[gg, t] = action
observations = torch.from_numpy(observations).float().to(device)
actions = torch.from_numpy(actions).float().unsqueeze(-1)
actions = torch.cat([torch.zeros(batch, 1, 1), actions], dim=1).to(device)
if net._get_name() == "SequentialVAE":
    _, _, m, m_t = net(observations, actions)
if net._get_name() == "SequentialVQVAE":
    _, _, m, m_t, _, _ = net(observations, actions)
if net._get_name() == "EMGFlowNet":
    latents, indices = net.sample_latents(observations, actions, rand_prob=0, prob_exponent=-1)
    # m, m_t = latents.chunk(2, dim=-1)
    m = latents

In [ ]:
# dataset = SLDataset(torch.cat([m, m_t], dim=-1).detach().reshape(-1, 4), labels.flatten())
dataset = SLDataset(m.detach().reshape(-1, 3), labels.flatten())
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)
snet = MLP(3, len(labels.unique()), [128, 128], activation=torch.nn.Tanh).to(device)
soptimizer = torch.optim.Adam(snet.parameters(), lr=1e-3)
for epoch in (pbar:=tqdm(range(100))):
    losses = 0.0
    for latent, label, _ in dataloader:
        out = snet(latent)
        loss = torch.nn.functional.cross_entropy(out, label.long().to(device))
        soptimizer.zero_grad()
        loss.backward()
        soptimizer.step()
        losses += loss.item()
    pbar.set_description(f"Loss: {losses:.4f}")

In [ ]:
# plt.plot((snet(torch.cat([m, m_t], dim=-1)).cpu().softmax(-1).argmax(-1) == labels).numpy().sum(0) / batch)
plt.plot((snet(m).cpu().softmax(-1).argmax(-1) == labels).numpy().sum(0) / batch)
# plt.plot(vqvaer)
# plt.legend(["GFlowNet", "VQ-VAE"])
# plt.ylim([0, 1]), plt.xlim([0, 99])